APT Intrusion Detection System — Starter Code
Based on: MAGIC (USENIX Security 2024) + FLASH (IEEE S&P 2024)
=============================================================
 
PROJECT OVERVIEW
----------------
Both papers build an IDS (Intrusion Detection System) that:
  1. Reads system audit logs
  2. Converts them into a "provenance graph" (nodes = system entities
     like processes/files/sockets; edges = events like read/write/connect)
  3. Uses a Graph Neural Network (GNN) to learn what "normal" looks like
  4. Flags anomalies as potential APT (Advanced Persistent Threat) attacks
 
YOUR DATASET: E3-THEIA (DARPA Transparent Computing)
------------------------------------------------------
The .pkl files store pre-processed provenance graphs as Python objects.
You have:
  - graphs_train_0.pkl, graphs_train_1.pkl, ... (training graphs)
  - graphs_test.pkl                              (test graphs)
 
HOW TO READ .pkl FILES
-----------------------
A .pkl (pickle) file is just a serialized Python object.
Think of it like a save-file for a Python variable.


In [ ]:
# Inspect .pkl files 

import pickle
import os

def inspect_pkl(filepath):
    """Load and inspect the contents of a .pkl file."""
    print(f"\n{'='*60}")
    print(f"Inspecting: {filepath}")
    print('='*60)

    with open(filepath, 'rb') as f:
        data = pickle.load(f)

    print(f"Type of loaded object: {type(data)}")

    # It's usually a list of graphs, or a dict
    if isinstance(data, list):
        print(f"Number of items (graphs): {len(data)}")
        # Peek at the first item
        first = data[0]
        print(f"Type of first item: {type(first)}")
        print(f"First item attributes: {dir(first)}")

        # PyTorch Geometric Data objects have these common fields:
        # .x          -> node feature matrix  [num_nodes, num_features]
        # .edge_index -> edge connectivity     [2, num_edges]
        # .edge_attr  -> edge features         [num_edges, num_edge_features]
        # .y          -> labels (0=benign, 1=malicious)
        # .num_nodes  -> count of nodes

        if hasattr(first, 'x'):
            print(f"\nNode features (x) shape: {first.x.shape}")
        if hasattr(first, 'edge_index'):
            print(f"Edge index shape:         {first.edge_index.shape}")
        if hasattr(first, 'edge_attr'):
            print(f"Edge attributes shape:    {first.edge_attr.shape}")
        if hasattr(first, 'y'):
            print(f"Labels (y):               {first.y}")
        if hasattr(first, 'num_nodes'):
            print(f"Number of nodes:          {first.num_nodes}")

    elif isinstance(data, dict):
        print(f"Keys: {list(data.keys())}")
        for k, v in data.items():
            print(f"  {k}: type={type(v)}, value preview={str(v)[:100]}")

    return data

In [2]:
# Change the path to wherever your .pkl files are located
DATA_DIR = "./data"

for fname in os.listdir(DATA_DIR):
    if fname.endswith('.pkl'):
        inspect_pkl(os.path.join(DATA_DIR, fname))



Inspecting: ./data/test0.pkl
Type of loaded object: <class 'dgl.heterograph.DGLGraph'>

Inspecting: ./data/train0.pkl
Type of loaded object: <class 'dgl.heterograph.DGLGraph'>

Inspecting: ./data/train1.pkl
Type of loaded object: <class 'dgl.heterograph.DGLGraph'>

Inspecting: ./data/train3.pkl
Type of loaded object: <class 'dgl.heterograph.DGLGraph'>

Inspecting: ./data/train2.pkl
Type of loaded object: <class 'dgl.heterograph.DGLGraph'>



Understand What the Models Actually Do

MAGIC vs FLASH — Key Differences
----------------------------------
 
MAGIC (your first model):
  - SELF-SUPERVISED: only trains on BENIGN data, no attack labels needed
  - Uses a "graph masked auto-encoder" (like BERT for graphs):
      * Randomly hides (masks) some node features
      * Trains GNN to reconstruct the hidden features
      * This forces the GNN to learn good representations
  - Detects anomalies using KNN (k-nearest neighbors) in embedding space:
      * Nodes/graphs that are far from all benign training examples = anomaly
  - Can do TWO levels of detection:
      * Graph-level: is this whole batch of logs malicious?
      * Node-level: which specific process/file is malicious?
 
FLASH (your second model):
  - Uses Word2Vec to encode SEMANTIC info (process names, file paths)
  - Uses GNN (GraphSAGE) to encode STRUCTURAL info (graph topology)
  - Combines both, then trains a lightweight XGBoost classifier
  - Key innovation: stores GNN embeddings in a database during training,
    reuses them at detection time → 3x faster than ThreaTrace
  - Does NODE-LEVEL detection
 
RANDOM FOREST in your dashboard:
  - This is the BASELINE/comparison model
  - Uses simpler features (not GNN embeddings) and a Random Forest classifier
  - Both papers compare against baselines like this
  - In FLASH specifically, they compared XGBoost vs Random Forest vs SVM
    (XGBoost won, but Random Forest was close)
 
So your dashboard has:
  MAGIC-inspired  → GNN autoencoder + KNN anomaly detection
  FLASH-inspired  → Word2Vec + GNN + XGBoost classifier
  Random Forest   → Simple feature extraction + Random Forest (baseline)

 
 

 
Project Structure Recommendation

NOTEBOOK (.ipynb) vs SCRIPTS (.py) — Which to use?
-----------------------------------------------------
Recommendation: USE BOTH
 
Use `.ipynb` (Jupyter notebooks) for:
  - Data exploration (inspecting `.pkl` files, visualizing graphs)
  - Training runs where you want to see output step by step
  - Generating plots and evaluation metrics for your dashboard
 
Use `.py` (Python scripts) for:
  - Reusable code (model definitions, data loaders, utilities)
  - Anything you import INTO your notebooks
 
Suggested folder structure:
```text
  project/
  ├── data/
  │   └── theia/
  │       ├── graphs_train_0.pkl
  │       ├── graphs_train_1.pkl
  │       ├── graphs_train_2.pkl
  │       ├── graphs_train_3.pkl
  │       └── graphs_test.pkl
  ├── notebooks/
  │   ├── 01_data_exploration.ipynb     ← inspect pkl, visualize graphs
  │   ├── 02_magic_model.ipynb          ← MAGIC training + evaluation
  │   ├── 03_flash_model.ipynb          ← FLASH training + evaluation
  │   └── 04_comparison_dashboard.ipynb ← compare all 3 models
  ├── models/
  │   ├── magic_model.py                ← MAGIC GNN architecture
  │   ├── flash_model.py                ← FLASH GNN + XGBoost
  │   └── baseline_rf.py                ← Random Forest baseline
  └── utils/
      ├── data_loader.py                ← load and batch the pkl files
      └── metrics.py                    ← precision, recall, F1, AUC
```


In [3]:
# Data Loading utility

import pickle
import os
from typing import List, Tuple

def load_theia_dataset(data_dir: str) -> Tuple[list, list]:
    """
    Load the pre-processed THEIA dataset from .pkl files.

    The MAGIC repo creates graphs with PyTorch Geometric Data objects.
    Each graph represents one "batch" of system logs.

    Returns:
        train_graphs: list of graph objects for training
        test_graphs:  list of graph objects for testing
    """
    train_graphs = []

    # Load all training pkl files (train_0, train_1, train_2, train_3)
    for i in range(4):  # adjust range if you have more/fewer files
        train_path = os.path.join(data_dir, f'graphs_train_{i}.pkl')
        if os.path.exists(train_path):
            with open(train_path, 'rb') as f:
                batch = pickle.load(f)
            if isinstance(batch, list):
                train_graphs.extend(batch)
            else:
                train_graphs.append(batch)
            print(f"Loaded train batch {i}: {len(batch) if isinstance(batch, list) else 1} graphs")

    # Load test set
    test_path = os.path.join(data_dir, 'graphs_test.pkl')
    with open(test_path, 'rb') as f:
        test_graphs = pickle.load(f)
    if not isinstance(test_graphs, list):
        test_graphs = [test_graphs]
    print(f"Loaded test set: {len(test_graphs)} graphs")

    print(f"\nTotal training graphs: {len(train_graphs)}")
    print(f"Total test graphs:     {len(test_graphs)}")
    return train_graphs, test_graphs


# MAGIC-inspired Model
 
MAGIC Architecture (simplified):
```text
  Input: Provenance graph G with node features X and edges E
       ↓
  [Masking] Randomly zero-out 50% of node feature vectors
       ↓
  [Graph Encoder - GAT layers] 
      Each node aggregates info from neighbors with attention weights
      Output: node embeddings h_n for every node
       ↓
  [Graph Pooling] Average all node embeddings → system-state embedding h_G
       ↓
  [Graph Decoder - reconstruction loss]
      Tries to reconstruct original masked features (teaches encoder)
       ↓
  At test time: compare h_G to training embeddings via KNN
      If distance > threshold θ → ANOMALY detected
```

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch_geometric.nn import GATConv, global_mean_pool
    from torch_geometric.data import Data, DataLoader

    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch/PyG not installed. Install with:")
    print("  pip install torch torch-geometric")


In [5]:
if TORCH_AVAILABLE:

    class MAGICEncoder(nn.Module):
        """
        Graph Attention Network encoder — the core of MAGIC.
        Learns node embeddings by aggregating neighbor information
        with learned attention weights.
        """
        def __init__(self, in_dim: int, hidden_dim: int, num_layers: int = 3):
            super().__init__()
            self.convs = nn.ModuleList()

            # First layer: input features → hidden
            self.convs.append(GATConv(in_dim, hidden_dim, heads=1))
            # Subsequent layers: hidden → hidden
            for _ in range(num_layers - 1):
                self.convs.append(GATConv(hidden_dim, hidden_dim, heads=1))

        def forward(self, x, edge_index):
            """
            x:          [num_nodes, in_dim]  — node feature matrix
            edge_index: [2, num_edges]        — COO format edge list
            Returns:    [num_nodes, hidden_dim] — node embeddings
            """
            h = x
            layer_outputs = [x]  # MAGIC concatenates all layer outputs

            for conv in self.convs:
                h = F.elu(conv(h, edge_index))
                layer_outputs.append(h)

            # Concatenate original features + all layer outputs (MAGIC design)
            node_embeddings = torch.cat(layer_outputs, dim=-1)
            return node_embeddings


    class MAGICDecoder(nn.Module):
        """
        Decoder that reconstructs masked node features.
        This provides the training signal (loss) for the encoder.
        """
        def __init__(self, embed_dim: int, out_dim: int):
            super().__init__()
            # Simple single GAT layer for reconstruction
            self.conv = GATConv(embed_dim, out_dim, heads=1)

        def forward(self, h, edge_index):
            return self.conv(h, edge_index)


    class MAGICModel(nn.Module):
        """
        Full MAGIC model: encoder + decoder + masking logic.

        Training: mask nodes → encode → decode → reconstruction loss
        Inference: encode → KNN anomaly detection (no masking)
        """
        def __init__(self, in_dim: int, hidden_dim: int = 64,
                    num_layers: int = 3, mask_rate: float = 0.5):
            super().__init__()
            self.mask_rate = mask_rate
            self.hidden_dim = hidden_dim
            self.num_layers = num_layers

            # After concatenating all layers, embedding size grows
            # in_dim + hidden_dim * num_layers
            embed_dim = in_dim + hidden_dim * num_layers

            self.encoder = MAGICEncoder(in_dim, hidden_dim, num_layers)
            self.decoder = MAGICDecoder(embed_dim, in_dim)

            # Learnable mask token (replaces masked node features)
            self.mask_token = nn.Parameter(torch.zeros(1, in_dim))

        def mask_nodes(self, x: torch.Tensor):
            """
            Randomly mask a fraction of nodes by replacing their
            features with the learnable mask token.
            Returns masked features and the mask indices.
            """
            num_nodes = x.size(0)
            num_masked = int(num_nodes * self.mask_rate)

            # Random permutation to pick which nodes to mask
            perm = torch.randperm(num_nodes, device=x.device)
            masked_indices = perm[:num_masked]
            unmasked_indices = perm[num_masked:]

            # Replace masked node features with mask token
            x_masked = x.clone()
            x_masked[masked_indices] = self.mask_token.expand(num_masked, -1)

            return x_masked, masked_indices, unmasked_indices

        def forward(self, x, edge_index, batch=None, training=True):
            """
            During training: apply masking, encode, decode, return embeddings + loss
            During inference: encode without masking, return embeddings only
            """
            if training:
                x_masked, masked_idx, _ = self.mask_nodes(x)
                # Encode with masked features
                node_embeddings = self.encoder(x_masked, edge_index)
                # Reconstruct original features of masked nodes
                recon = self.decoder(node_embeddings, edge_index)
                # Scaled cosine loss (from the paper)
                loss = self._cosine_loss(x[masked_idx], recon[masked_idx])
                # System-state embedding = average of all node embeddings
                if batch is not None:
                    graph_embedding = global_mean_pool(node_embeddings, batch)
                else:
                    graph_embedding = node_embeddings.mean(0, keepdim=True)
                return graph_embedding, node_embeddings, loss
            else:
                # No masking at inference time
                node_embeddings = self.encoder(x, edge_index)
                if batch is not None:
                    graph_embedding = global_mean_pool(node_embeddings, batch)
                else:
                    graph_embedding = node_embeddings.mean(0, keepdim=True)
                return graph_embedding, node_embeddings

        def _cosine_loss(self, original, reconstructed, gamma=3):
            """
            Scaled cosine error loss (Equation from MAGIC paper).
            gamma=3 makes the loss penalize hard examples more.
            """
            # Cosine similarity between original and reconstructed features
            cos_sim = F.cosine_similarity(original, reconstructed, dim=-1)
            # Loss = (1 - cosine_similarity)^gamma  — scaled to emphasize hard samples
            loss = (1 - cos_sim).pow(gamma).mean()
            return loss


    class MAGICDetector:
        """
        KNN-based anomaly detector — the detection module of MAGIC.

        During training: memorize all benign graph/node embeddings
        During inference: find k nearest neighbors, compute anomaly score
        """
        def __init__(self, k: int = 10, threshold: float = 2.0):
            self.k = k
            self.threshold = threshold
            self.benign_embeddings = None  # stored training embeddings

        def fit(self, embeddings: torch.Tensor):
            """Store benign embeddings from training data."""
            self.benign_embeddings = embeddings.detach().cpu()
            print(f"Detector fitted with {len(embeddings)} benign embeddings")

        def predict(self, embeddings: torch.Tensor):
            """
            Compute anomaly scores for new embeddings.
            Score = average distance to k nearest benign neighbors,
                    normalized by average training distance.
            Returns: anomaly_scores, predictions (0=benign, 1=anomaly)
            """
            emb = embeddings.detach().cpu()
            scores = []

            # Compute pairwise distances (could use scipy KD-tree for speed)
            for e in emb:
                dists = torch.norm(self.benign_embeddings - e.unsqueeze(0), dim=1)
                knn_dists, _ = torch.topk(dists, self.k, largest=False)
                scores.append(knn_dists.mean().item())

            scores = torch.tensor(scores)
            # Normalize by average training distance
            avg_train_dist = self._avg_training_distance()
            normalized_scores = scores / (avg_train_dist + 1e-8)

            predictions = (normalized_scores >= self.threshold).long()
            return normalized_scores, predictions

        def _avg_training_distance(self):
            """Average KNN distance within training embeddings (used for normalization)."""
            total = 0
            n = min(100, len(self.benign_embeddings))  # sample for efficiency
            for i in range(n):
                dists = torch.norm(self.benign_embeddings -
                                self.benign_embeddings[i].unsqueeze(0), dim=1)
                dists[i] = float('inf')  # exclude self
                knn_dists, _ = torch.topk(dists, self.k, largest=False)
                total += knn_dists.mean().item()
            return total / n

# FLASH-inspired Model

FLASH Architecture (simplified):
```text
  Input: Provenance graph with rich node attributes (names, paths, IPs)
       ↓
  [Word2Vec encoder]
      Convert node attributes (process name, file path, etc.) into
      dense semantic vectors. Add positional encoding for temporal order.
       ↓
  [GraphSAGE encoder]
      Use word2vec vectors as node features.
      Aggregate neighborhood structure into node embeddings.
       ↓
  [Embedding Database]
      Store GNN embeddings keyed by "Persistent Node ID"
      At runtime, reuse stored embeddings (avoids re-running GNN)
       ↓
  [XGBoost classifier]
      Concatenate Word2Vec + GNN embeddings.
      Train to classify node types (process/file/socket).
      At runtime, MISCLASSIFICATION = anomaly!
```
 
### WHY IS MISCLASSIFICATION AN ANOMALY?

  The model learns "a process node looks like X, a file node looks like Y".

  Malicious nodes behave unusually → their embedding doesn't match any known node type cleanly → classifier is uncertain/wrong → flagged as malicious.


In [6]:
try:
    from torch_geometric.nn import SAGEConv
    import numpy as np
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import classification_report, roc_auc_score, f1_score

    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("scikit-learn not installed. Install with: pip install scikit-learn")

In [7]:
if TORCH_AVAILABLE:

    class FLASHGraphSAGE(nn.Module):
        """
        GraphSAGE encoder — the structural encoder in FLASH.
        Aggregates neighborhood features into node representations.
        """
        def __init__(self, in_dim: int, hidden_dim: int, out_dim: int,
                    num_layers: int = 2, dropout: float = 0.3):
            super().__init__()
            self.convs = nn.ModuleList()
            self.convs.append(SAGEConv(in_dim, hidden_dim))
            for _ in range(num_layers - 2):
                self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.convs.append(SAGEConv(hidden_dim, out_dim))
            self.dropout = dropout
            # Classification head (node type prediction)
            # FLASH trains on node type (process/file/socket) classification
            # Anomalies are nodes that don't fit learned type distributions

        def forward(self, x, edge_index):
            h = x
            for i, conv in enumerate(self.convs):
                h = conv(h, edge_index)
                if i < len(self.convs) - 1:
                    h = F.relu(h)
                    h = F.dropout(h, p=self.dropout, training=self.training)
            return h  # [num_nodes, out_dim]


    class SimpleWord2VecEncoder:
        """
        Simplified Word2Vec-like encoder for node attributes.
        In the real FLASH, this uses a proper Word2Vec model trained on logs.
        Here we simulate it with a hash-based lookup embedding.

        In practice you would:
        1. Extract text attributes per node (process names, file paths)
        2. Train Word2Vec on a corpus of benign log sentences
        3. Look up embeddings for each token and average them
        4. Add positional encoding for temporal order
        """
        def __init__(self, embed_dim: int = 64):
            self.embed_dim = embed_dim
            # Simulated: map string attributes to fixed vectors
            self._cache = {}

        def encode_node(self, attributes: dict) -> np.ndarray:
            """
            Encode a node's semantic attributes into a vector.
            attributes: dict with keys like 'name', 'path', 'ip', 'port'
            """
            # Combine all attribute strings
            text = ' '.join(str(v) for v in attributes.values())
            if text not in self._cache:
                # In real FLASH: look up Word2Vec embeddings for each token
                # Here: deterministic hash for reproducibility
                rng = np.random.RandomState(hash(text) % (2**31))
                self._cache[text] = rng.randn(self.embed_dim).astype(np.float32)
            return self._cache[text]


class FLASHDetector:
    """
    Full FLASH detection pipeline.
    Combines GNN structural embeddings + semantic Word2Vec embeddings,
    then uses XGBoost (or Random Forest) to classify node types.
    Misclassified nodes are flagged as anomalies.
    """
    def __init__(self, use_xgboost: bool = True):
        self.use_xgboost = use_xgboost
        self.embedding_database = {}  # Persistent Node ID → stored embedding
        self.classifier = None
        self.label_encoder = LabelEncoder() if SKLEARN_AVAILABLE else None

    def build_embedding_database(self, gnn_model, train_graphs):
        """
        Run GNN on training graphs and store embeddings keyed by node ID.
        This is the "embedding recycling database" from FLASH.
        At inference time, we reuse these instead of re-running GNN.
        """
        print("Building embedding database from training graphs...")
        if not TORCH_AVAILABLE:
            return

        gnn_model.eval()
        with torch.no_grad():
            for graph in train_graphs:
                embeddings = gnn_model(graph.x, graph.edge_index)
                # Store by node persistent ID (here simplified to node index)
                # In real FLASH, the key is based on process name/file path
                for i, emb in enumerate(embeddings):
                    node_key = f"node_{i}"  # real FLASH uses PNI
                    self.embedding_database[node_key] = emb.numpy()

        print(f"Database contains {len(self.embedding_database)} node embeddings")

    def train_classifier(self, X_train, y_train):
        """
        Train XGBoost (or Random Forest) classifier on combined embeddings.
        y_train = node type labels (process=0, file=1, socket=2)
        """
        if not SKLEARN_AVAILABLE:
            return

        if self.use_xgboost:
            try:
                from xgboost import XGBClassifier
                self.classifier = XGBClassifier(
                    n_estimators=100, max_depth=6,
                    use_label_encoder=False, eval_metric='mlogloss'
                )
            except ImportError:
                print("XGBoost not available, falling back to GradientBoosting")
                self.classifier = GradientBoostingClassifier(n_estimators=100)
        else:
            self.classifier = RandomForestClassifier(n_estimators=100, n_jobs=-1)

        self.classifier.fit(X_train, y_train)
        print("Classifier trained.")

    def predict_anomalies(self, X_test, threshold: float = 0.5):
        """
        Detect anomalies: nodes where predicted node-type probability
        is below threshold → uncertain → potentially malicious.
        """
        if self.classifier is None:
            raise RuntimeError("Call train_classifier first")

        proba = self.classifier.predict_proba(X_test)
        max_confidence = proba.max(axis=1)  # confidence in best prediction
        # Low confidence = node doesn't fit known benign patterns = anomaly
        anomaly_scores = 1 - max_confidence
        predictions = (anomaly_scores >= threshold).astype(int)
        return anomaly_scores, predictions

In [8]:
# Random Forest Baseline

class RandomForestBaseline:
    """
    Baseline model: simple hand-crafted features + Random Forest.
    No GNN — just statistics about each node's connectivity.

    This is what MAGIC and FLASH are compared against.
    Expected to perform WORSE than the GNN-based approaches.
    """
    def __init__(self, n_estimators: int = 100):
        self.rf = RandomForestClassifier(n_estimators=n_estimators,
                                        n_jobs=-1) if SKLEARN_AVAILABLE else None

    def extract_features(self, graph) -> np.ndarray:
        """
        Extract simple graph statistics per node as features.
        These are much simpler than what MAGIC/FLASH learn.
        """
        if not TORCH_AVAILABLE:
            return np.array([])

        x = graph.x.numpy()  # raw node features
        edge_index = graph.edge_index.numpy()
        num_nodes = x.shape[0]

        features = []
        for i in range(num_nodes):
            # In-degree: how many edges point TO this node
            in_degree = (edge_index[1] == i).sum()
            # Out-degree: how many edges point FROM this node
            out_degree = (edge_index[0] == i).sum()
            # Raw features + degree info
            node_feat = np.concatenate([
                x[i],
                [in_degree, out_degree, in_degree + out_degree]
            ])
            features.append(node_feat)

        return np.array(features)

    def fit(self, train_graphs, labels):
        all_features = []
        all_labels = []
        for graph, label in zip(train_graphs, labels):
            feats = self.extract_features(graph)
            all_features.append(feats.mean(axis=0))  # graph-level: average
            all_labels.append(label)
        if SKLEARN_AVAILABLE:
            self.rf.fit(np.array(all_features), np.array(all_labels))

    def predict(self, test_graphs):
        all_features = []
        for graph in test_graphs:
            feats = self.extract_features(graph)
            all_features.append(feats.mean(axis=0))
        if SKLEARN_AVAILABLE:
            return self.rf.predict(np.array(all_features))
        return np.zeros(len(test_graphs))

In [9]:
# Evaluation Metrics

def evaluate_model(y_true, y_pred, y_scores=None, model_name="Model"):
    """
    Compute and print detection metrics.
    These are the metrics both papers report (Table 3 in MAGIC, Table 2 in FLASH).
    """
    if not SKLEARN_AVAILABLE:
        print("scikit-learn required for evaluation")
        return {}

    from sklearn.metrics import (precision_score, recall_score,
                                f1_score, roc_auc_score,
                                confusion_matrix)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    print(f"\n{'='*40}")
    print(f"  {model_name} Results")
    print(f"{'='*40}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  FPR:       {fpr:.4f}")
    print(f"  TP={tp}, FP={fp}, FN={fn}, TN={tn}")

    if y_scores is not None:
        try:
            auc = roc_auc_score(y_true, y_scores)
            print(f"  AUC-ROC:   {auc:.4f}")
        except ValueError:
            pass 
    return {
        'precision': precision, 'recall': recall,
        'f1': f1, 'fpr': fpr, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn
    }

In [10]:
# Full Training Loop (MAGIC)

def train_magic(train_graphs, test_graphs, in_dim: int,
                epochs: int = 50, hidden_dim: int = 64, lr: float = 0.001):
    """
    Complete MAGIC training and evaluation pipeline.

    Args:
        train_graphs: list of PyG Data objects (BENIGN ONLY)
        test_graphs:  list of PyG Data objects (benign + attack)
        in_dim:       number of node features
        epochs:       training epochs
    """
    if not TORCH_AVAILABLE:
        print("PyTorch required")
        return

    model = MAGICModel(in_dim=in_dim, hidden_dim=hidden_dim, num_layers=3)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    detector = MAGICDetector(k=10, threshold=2.0)

    # ---- TRAINING PHASE ----
    print("Training MAGIC...")
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for graph in train_graphs:
            optimizer.zero_grad()
            graph_emb, node_emb, loss = model(
                graph.x, graph.edge_index,
                batch=getattr(graph, 'batch', None),
                training=True
            )
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_graphs):.4f}")

    # ---- FIT DETECTOR ----
    # Collect benign graph-level embeddings for KNN memory
    model.eval()
    benign_embeddings = []
    with torch.no_grad():
        for graph in train_graphs:
            graph_emb, _ = model(graph.x, graph.edge_index, training=False)
            benign_embeddings.append(graph_emb.squeeze(0))
    detector.fit(torch.stack(benign_embeddings))

    # ---- INFERENCE ----
    print("\nRunning detection on test set...")
    test_embeddings = []
    with torch.no_grad():
        for graph in test_graphs:
            graph_emb, _ = model(graph.x, graph.edge_index, training=False)
            test_embeddings.append(graph_emb.squeeze(0))
    test_embs = torch.stack(test_embeddings)
    scores, preds = detector.predict(test_embs)

    # Ground truth labels (1 = malicious graph, 0 = benign)
    y_true = [int(graph.y.item()) for graph in test_graphs]

    return evaluate_model(y_true, preds.numpy(), scores.numpy(), "MAGIC")

## QUICK START GUIDE

1. First, inspect your data:
Uncomment the inspect_pkl block at the top (STEP 0) and run: `python apt_ids_project_guide.py`
OR in a notebook:
```python
import pickle
with open('data/theia/graphs_test.pkl', 'rb') as f:
data = pickle.load(f)
print(type(data), len(data))
print(data[0])  # look at first graph
```

2. Install dependencies:
```bash
pip install torch torchvision torchaudio
pip install torch-geometric
pip install scikit-learn xgboost networkx matplotlib
```

3. Run the models:
```python
from apt_ids_project_guide import (
     load_theia_dataset, MAGICModel, MAGICDetector,
     FLASHDetector, RandomForestBaseline, evaluate_model
     )
train_graphs, test_graphs = load_theia_dataset('data/theia')
# Then call train_magic(train_graphs, test_graphs, in_dim=???)
# (in_dim comes from inspecting your data first!)
```

4. For your dashboard:
Collect the metrics dict returned by `evaluate_model()` for each of the three models, then plot them side by side.

NOTE ON THE THEIA DATASET:
* Training graphs are BENIGN ONLY (MAGIC trains unsupervised)
* Test graphs contain BOTH benign and attack graphs
* The `.y` field on each graph object should be 0 (benign) or 1 (attack)
* THEIA has ~1.6M benign nodes and ~25K malicious nodes
